In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import subprocess
import sys
from sklearn.preprocessing import MinMaxScaler

# Install pgeocode for geographic distance calculation
try:
    import pgeocode
except ImportError:
    print("Installing pgeocode...")
    subprocess.run([sys.executable, "-m", "pip", "install", "pgeocode"], check=True)
    import pgeocode

try:
    from ethnicolr import census_ln
except ImportError:
    print("Installing ethnicolr...")
    # Note: ethnicolr requires TensorFlow. This installation might take a moment.
    subprocess.run([sys.executable, "-m", "pip", "install", "ethnicolr"], check=True)
    from ethnicolr import census_ln

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import pgeocode
import joblib


In [6]:
#file locations
parquet_file_paths={
    "patient": r"Client_Data_files\Parquets\synthetic_patients.parquet",
    "encounter": r"Client_Data_files\Parquets\synthetic_encounters.parquet",
    "hospitals": r"Client_Data_files\Parquets\synthetic_hospitals.parquet",
    "provider": r"Client_Data_files\Parquets\synthetic_providers.parquet",    
}

# Reading the parquet files
patient_df = pd.read_parquet(parquet_file_paths['patient'])
encounter_df = pd.read_parquet(parquet_file_paths['encounter'])
hospital_df = pd.read_parquet(parquet_file_paths['hospitals'])
provider_df = pd.read_parquet(parquet_file_paths['provider'])

print("Dataframes loaded successfully.")
print(f"Patient DF shape: {patient_df.shape}")
print(f"Encounter DF shape: {encounter_df.shape}")
print(f"Provider DF shape: {provider_df.shape}")
print(f"Hospital DF shape: {hospital_df.shape}")

Dataframes loaded successfully.
Patient DF shape: (100000, 16)
Encounter DF shape: (200000, 17)
Provider DF shape: (5000, 19)
Hospital DF shape: (200, 14)


In [7]:
# Creating a data map for easy access
data_map={
    "patient": patient_df,
    "encounter": encounter_df,
    "hospitals": hospital_df,
    "provider": provider_df
}

for key, df in data_map.items():
    print(f"Dataframe: {key}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")    
    for col in df.columns:
        if df[col].isna().sum() > 0:
            print(f"Column '{col}' has {df[col].isna().sum()} missing values.")
    print("")

Dataframe: patient
Shape: (100000, 16)
Columns: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'primary_language', 'zip_code', 'insurance_type', 'household_income', 'education_level', 'age', 'cultural_background', 'preferred_provider_language', 'cultural_preferences']

Dataframe: encounter
Shape: (200000, 17)
Columns: ['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']

Dataframe: hospitals
Shape: (200, 14)
Columns: ['hospital_id', 'hospital_name', 'hospital_type', 'zip_code', 'bed_count', 'teaching_hospital', 'trauma_center', 'language_services_available', 'cultural_competency_program', 'interpreter_services_24_7', 'community_health_programs', 'overall_rating

In [8]:
patient_df_temp=pd.DataFrame()
patient_df_temp=patient_df[patient_df['race']=='Hispanic or Latino'][['patient_id','first_name','last_name','race','ethnicity']].copy()
patient_race_pred=census_ln(patient_df_temp, 'last_name')

2025-09-29 00:59:46,533 - INFO - Preserving 12965 duplicate rows based on column 'last_name'
2025-09-29 00:59:46,538 - INFO - Data filtering summary: 12997 → 12997 rows (kept 100.0%)
2025-09-29 00:59:46,550 - INFO - Loading Census 2000 data from c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\ethnicolr\data\census\census_2000.csv...
2025-09-29 00:59:46,726 - INFO - Loaded 151670 last names from Census 2000
2025-09-29 00:59:46,726 - INFO - Merging demographic data for 12997 records...
2025-09-29 00:59:46,789 - INFO - Matched 12997 of 12997 rows (100.0%)
2025-09-29 00:59:46,789 - INFO - Added columns: pct2prace, pctaian, pctapi, pctblack, pcthispanic, pctwhite


In [9]:
race_mapping={
    'white': 'White',
    'black': 'Black or African American',
    'api': 'Asian',    
    'aian': 'Native American',
    '2prace': 'Other'
}

In [10]:
race_cols=['pctwhite','pctblack','pctapi','pctaian','pct2prace']
patient_race_pred['derived_race'] = patient_race_pred[race_cols].idxmax(axis=1).str.replace('pct', '').map(race_mapping)

# Create a mapping from patient_id to derived_race
id_to_derived_race = dict(zip(patient_race_pred['patient_id'], patient_race_pred['derived_race']))

# Update the race column only for Hispanic or Latino patients
patient_df.loc[patient_df['race'] == 'Hispanic or Latino', 'race'] = \
    patient_df.loc[patient_df['race'] == 'Hispanic or Latino', 'patient_id'].map(id_to_derived_race)

In [11]:
provider_race_predictions = census_ln(provider_df, 'last_name')

# Derive race for the provider_df as it is missing from the source data
print("Deriving race for providers from last names...")
race_cols = ['pctwhite','pctblack','pctapi','pctaian','pct2prace']
provider_df['provider_race'] = provider_race_predictions[race_cols].idxmax(axis=1).str.replace('pct', '').map(race_mapping)
print("Provider race derivation complete.")

# Deriving provider ethnicity from the race_predictions
print("Deriving ethnicity for providers from race predictions...")
provider_df['provider_ethnicity'] = provider_race_predictions['pcthispanic'].apply(lambda x: 'Hispanic or Latino' if float(x) >= 50 else 'Not Hispanic or Latino')
print("Provider ethnicity derivation complete.")


2025-09-29 00:59:46,856 - INFO - Preserving 4968 duplicate rows based on column 'last_name'
2025-09-29 00:59:46,856 - INFO - Data filtering summary: 5000 → 5000 rows (kept 100.0%)
2025-09-29 00:59:46,866 - INFO - Merging demographic data for 5000 records...
2025-09-29 00:59:46,906 - INFO - Matched 5000 of 5000 rows (100.0%)
2025-09-29 00:59:46,906 - INFO - Added columns: pct2prace, pctaian, pctapi, pctblack, pcthispanic, pctwhite


Deriving race for providers from last names...
Provider race derivation complete.
Deriving ethnicity for providers from race predictions...
Provider ethnicity derivation complete.


In [12]:
for key, df in data_map.items():
    print(f"Dataframe: {key}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")    
    for col in df.columns:
        if df[col].isna().sum() > 0:
            print(f"Column '{col}' has {df[col].isna().sum()} missing values.")
    print("")

Dataframe: patient
Shape: (100000, 16)
Columns: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'primary_language', 'zip_code', 'insurance_type', 'household_income', 'education_level', 'age', 'cultural_background', 'preferred_provider_language', 'cultural_preferences']

Dataframe: encounter
Shape: (200000, 17)
Columns: ['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']

Dataframe: hospitals
Shape: (200, 14)
Columns: ['hospital_id', 'hospital_name', 'hospital_type', 'zip_code', 'bed_count', 'teaching_hospital', 'trauma_center', 'language_services_available', 'cultural_competency_program', 'interpreter_services_24_7', 'community_health_programs', 'overall_rating

In [18]:
def get_recommendations(patient_id, required_specialty, 
                        # DataFrames
                        all_providers_df, 
                        all_patients_df, 
                        all_hospitals_df,
                        # Trained Models & Scaling Data
                        models_dict,
                        training_min_dist, 
                        training_max_dist):
    """
    Generates ranked doctor recommendations using the segmented model strategy.
    
    Args:
        patient_id (int): The ID of the patient seeking a recommendation.
        required_specialty (str): The medical specialty required.
        all_providers_df (pd.DataFrame): The full provider dataframe.
        all_patients_df (pd.DataFrame): The full patient dataframe.
        all_hospitals_df (pd.DataFrame): The full hospital dataframe.
        models_dict (dict): The dictionary of trained models for each preference segment.
        training_min_dist (float): The minimum distance calculated from the full training set.
        training_max_dist (float): The maximum distance calculated from the full training set.
        
    Returns:
        pd.DataFrame: A ranked dataframe of recommended providers.
    """
    print(f"\n--- Starting Recommendation Phase for Patient ID: {patient_id} ---")
    
    # --- Step 1: Patient Lookup & Model Routing ---
    patient_info = all_patients_df[all_patients_df['patient_id'] == patient_id]
    if patient_info.empty:
        return "Error: Patient ID not found."
        
    preference = patient_info['cultural_preferences'].iloc[0]
    model_to_use = None
    model_name = ""

    # Determine which trained model to use based on the patient's preference
    if preference == 'Culturally Similar Provider; Same Language Provider':
        model_name = 'Culturally Similar Provider; Same Language Provider' # Match the exact key from training
    elif preference == 'Culturally Similar Provider':
        model_name = 'Culturally Similar Provider'
    elif preference == 'Same Language Provider':
        model_name = 'Same Language Provider'
    else: # Fallback for any other preferences, like just "Same Language Provider"
        model_name = 'No Specific Preference'

    model_to_use = models_dict.get(model_name)
    print(f"Patient preference: '{preference}'. Routing to '{model_name}' model.")
        
    if not model_to_use:
        return f"Error: Model for preference group '{model_name}' was not trained (likely due to small size)."


    # --- Step 2: Candidate Generation (Hard Filters) ---
    candidate_providers_accepting_new = all_providers_df[all_providers_df['accepts_new_patients'] == True]
    previous_provider=encounter_df[encounter_df['patient_id']==patient_id]['provider_id'].unique()
    candidate_providers_not_accepting_new = all_providers_df[
        (all_providers_df['accepts_new_patients'] == False) &
        (all_providers_df['provider_id'].isin(previous_provider))
    ]
    
    candidate_providers_all=pd.concat([candidate_providers_accepting_new,candidate_providers_not_accepting_new])

    candidate_providers = candidate_providers_all[
        (candidate_providers_all['specialty'] == required_specialty)         
    ].copy()
    if candidate_providers.empty:
        print(f"No providers found for specialty '{required_specialty}' ")
        candidate_providers = candidate_providers_all

    # --- Step 3: Feature Engineering for Inference ---
    # Merge all necessary info for the patient and candidate providers
    inference_df = candidate_providers.assign(key=1).merge(patient_info.assign(key=1), on='key').drop('key', axis=1)
    inference_df = pd.merge(inference_df, all_hospitals_df, left_on='hospital_affiliation', right_on='hospital_id', how='left')

    #print(f'inference_df columns: {inference_df.columns.tolist()}')

    # Re-create the exact same features used in training
    inference_df.rename(columns={'cultural_competency_rating_y': 'cultural_competency_rating_prov'}, inplace=True)
    inference_df['race_match'] = (inference_df['race'] == inference_df['provider_race']).astype(int)
    inference_df['ethnicity_match'] = (inference_df['ethnicity'] == inference_df['provider_ethnicity']).astype(int)

    # create language match if preferred_provider_language in languages_spoken (semicolon-separated string)
    def language_match_func(row):
        if pd.isna(row['languages_spoken']) or pd.isna(row['preferred_provider_language']):
            return 0
        spoken = [lang.strip() for lang in str(row['languages_spoken']).split(';')]
        return 1 if row['preferred_provider_language'] in spoken else 0

    inference_df['language_match'] = inference_df.apply(language_match_func, axis=1)

    # Geographic Feature - CRITICAL: Use the same scaling as the training data
    dist = pgeocode.GeoDistance('US')
    inference_df['distance_km'] = dist.query_postal_code(
        inference_df['zip_code_x'].astype(str).tolist(), 
        inference_df['zip_code_y'].astype(str).tolist()
    )
    # Impute missing distances using the same logic (specialty mean, then global mean)
    mean_dist_by_specialty = inference_df.groupby('specialty')['distance_km'].transform('mean')
    inference_df['distance_km'].fillna(mean_dist_by_specialty, inplace=True)
    inference_df['distance_km'].fillna(training_max_dist / 2, inplace=True) # Fallback with a reasonable value

    inference_df['proximity_score'] = 1 - (
        (inference_df['distance_km'] - training_min_dist) / (training_max_dist - training_min_dist)
    )
    # Clip scores to be between 0 and 1, in case a new distance is outside the training range
    inference_df['proximity_score'] = inference_df['proximity_score'].clip(0, 1)

    inference_df['cultural_competency_rating_prov'] = inference_df['cultural_competency_rating']
    # --- Step 4: Predict Scores ---
    # Ensure the feature list matches the one used for training
    features = [
        'years_experience', 'cultural_competency_rating_prov', 'communication_rating',
        'race_match', 'ethnicity_match', 'language_match', 'proximity_score', 
        'interpreter_services_24_7'
    ]
    X_inference = inference_df[features]
    predicted_scores = model_to_use.predict(X_inference)
    inference_df['predicted_success_score'] = predicted_scores

    # --- Step 5: Rank and Return ---
    recommendations = inference_df.sort_values(by='predicted_success_score', ascending=False)

    print("--- Recommendations Generated ---")
   # print("--- Recommendations Generated ---")
    return recommendations[['provider_id', 'first_name_x', 'last_name_x', 'specialty', 'hospital_name','distance_km', 'predicted_success_score']]



In [14]:
# --- 2. Load the saved training artifacts ---
file_path = r'RandomForestRegressor_CompositeSuccessScore_artifacts.joblib'
loaded_artifacts = joblib.load(file_path)

# Unpack the artifacts into variables
models_dict = loaded_artifacts['models']
training_min_dist = loaded_artifacts['min_dist']
training_max_dist = loaded_artifacts['max_dist']

print("--- Training artifacts loaded successfully ---")
print(f"Loaded {len(models_dict)} model(s).")

--- Training artifacts loaded successfully ---
Loaded 4 model(s).


In [19]:
# --- 3. Now you can use your get_recommendations function ---
# (Make sure the function definition is present in this script)

# Example Usage:
example_patient_id = 'PAT_046599'
example_specialty = 'Cardiology'

final_recommendations = get_recommendations(
    patient_id=example_patient_id,
    required_specialty=example_specialty,
    all_providers_df=provider_df,
    all_patients_df=patient_df,
    all_hospitals_df=hospital_df,
    models_dict=models_dict, # Using the loaded models
    training_min_dist=training_min_dist, # Using the loaded value
    training_max_dist=training_max_dist  # Using the loaded value
)

display(final_recommendations.head(5))


--- Starting Recommendation Phase for Patient ID: PAT_046599 ---
Patient preference: 'Culturally Similar Provider'. Routing to 'Culturally Similar Provider' model.
--- Recommendations Generated ---


,provider_id,first_name_x,last_name_x,specialty,hospital_name,distance_km,predicted_success_score
199,PROV_03619,Mary,Mohamed,Cardiology,Good Samaritan Medical Center,3873.374542,0.873326
49,PROV_00835,Ana,Smith,Cardiology,Community Health System,2686.917181,0.872959
242,PROV_04354,Robert,Anderson,Cardiology,Community Medical Center,2686.917181,0.870991
98,PROV_01860,Ming,Martinez,Cardiology,General Hospital,2686.917181,0.870412
84,PROV_01606,Omar,Garcia,Cardiology,Good Samaritan Medical Center,3873.374542,0.869985
